# Aşama 1: Veri Seti Keşfi ve Şema Analizi

Bu çalışma defteri **Stanford Energy Control Laboratory (INR21700-M50T)** ana veri seti ile **NASA Ames PCoE (18650)** yardımcı veri setinin dosya yapılarını, sütunlarını, birimlerini, örnekleme sıklıklarını, eksik değerlerini ve fiziksel sınır anomalilerini inceler.

> **Önemli Kural:** Stanford ve NASA verileri farklı kimya, kapasite ve test koşullarına sahip olduğu için aynı batarya gibi birleştirilmez; ayrı şema tablolarında raporlanır.

In [ ]:
from pathlib import Path
import sys

# Yerel çalışma veya Google Colab ortamı için proje kök dizinini ekle
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import (
    STANFORD_CELL_CRATES,
    summarize_stanford_directory,
    summarize_nasa_directory,
    load_stanford_diagnostic,
    extract_stanford_diag_df,
    load_nasa_mat,
)

DATA_RAW = PROJECT_ROOT / 'data' / 'raw'
TABLES_DIR = PROJECT_ROOT / 'results' / 'tables'
TABLES_DIR.mkdir(parents=True, exist_ok=True)
print('Proje kökü:', PROJECT_ROOT)

## 1. Stanford INR21700-M50T Şema ve Kalite Denetimi
Hücre bazlı şarj C-oranları (`README.xlsx` doğrulaması): `W4, W7, V4 -> C/4`; `W5, W8 -> C/2`; `W9, V5 -> 1C`; `W3, W10, G1 -> 3C`.

In [ ]:
stanford_summary = summarize_stanford_directory(DATA_RAW / 'stanford')
stanford_csv = TABLES_DIR / 'stanford_schema_summary.csv'
stanford_summary.to_csv(stanford_csv, index=False)
print(f'Stanford özet satır sayısı: {len(stanford_summary)} -> {stanford_csv}')
stanford_summary

## 2. NASA 18650 Yardımcı Veri Seti Şema ve Kalite Denetimi
`B0005`, `B0006`, `B0007`, `B0018` (24 °C), `B0029` (43 °C) ve `B0053` (4 °C) hücrelerinin döngü sayıları, kapasite düşüşleri, empedans aralığı (`Re`, `Rct`) ve ölçüm kalitesi incelenir.

In [ ]:
nasa_summary = summarize_nasa_directory(DATA_RAW / 'nasa')
nasa_csv = TABLES_DIR / 'nasa_schema_summary.csv'
nasa_summary.to_csv(nasa_csv, index=False)
print(f'NASA özet satır sayısı: {len(nasa_summary)} -> {nasa_csv}')
nasa_summary

## 3. Eğitim / Kalibrasyon ve Bağımsız Doğrulama Ayrımı (Veri Sızıntısız)
- **Kalibrasyon / Eğitim Hücreleri:** `W4` (C/4), `W8` (C/2), `W9` (1C), `W10` (3C) — `Diag_1` ve `Cycling_1`
- **Bağımsız Doğrulama Hücreleri:** `W5` (C/2) ve `V4` (C/4) — kalibrasyona hiç dahil edilmeyen hücreler + ileri çevrimler
- **Bağımsız Nitel Kontrol (NASA):** `B0005` (24 °C), `B0029` (43 °C), `B0053` (4 °C) — farklı hücre/kimya altında sıcaklık ve yaşlanma eğilimi kontrolü.

In [ ]:
# Örnek kapasite kaybı özeti (Stanford W8 vs. bağımsız W5)
cap_data = load_stanford_diagnostic(DATA_RAW / 'stanford' / 'diagnostic' / 'capacity_test.mat')
for cid in ['W4', 'W8', 'W9', 'W10', 'W5']:
    df_d1 = extract_stanford_diag_df(cap_data, cid, 1)
    print(f'{cid} ({STANFORD_CELL_CRATES[cid]}): Diag_1 max capacity = {df_d1["capacity_Ah"].max():.4f} Ah, N={len(df_d1)}')